# Backorder Action Assistant — Data Pipeline & Scoring Logic

This notebook walks through the actual code behind the POC: cleaning the raw Power BI export, generating
the SAP-style mock fields, computing the prioritization score, and the AI drill-down logic used in the demo.

**Inputs:** `PTO Report.xlsx` (open_vendor_orders sheet) and `ASN Dashboard.xlsx` (Suppliers sheet) — both
direct exports from the client's Power BI model.

## Step 1 — Load & Clean the Raw Export

The raw sheet reports over 1,000,000 rows, but that's a Power BI charting artifact: each real backorder
is repeated once per calendar day (for date-axis trend charts). We recover the true records using the
internal key Power BI itself uses to mark the representative row.

In [1]:
import pandas as pd
import numpy as np

PTO_PATH = "PTO_Report_sample.xlsx"   # PTO Report.xlsx (open_vendor_orders sheet)
ASN_PATH = "ASN_Dashboard_sample.xlsx"  # ASN Dashboard.xlsx (Suppliers sheet)

raw = pd.read_excel(PTO_PATH, sheet_name="open_vendor_orders")
print("Raw sheet shape (includes blank formatting rows):", raw.shape)

# Drop fully-blank formatting rows
raw = raw.dropna(subset=["Key"]).copy()
print("Rows with real data:", raw.shape)


Raw sheet shape (includes blank formatting rows): (1048575, 38)


Rows with real data: (89705, 38)


In [2]:
# Each real backorder is exploded into one row per calendar day.
# open862_key is the true record id; Lowest Open Qty Flag / Max_Z_Effective_Date
# mark the representative snapshot row (the same logic Power BI itself uses).
raw_sorted = raw.sort_values(
    ["open862_key", "Lowest Open Qty Flag", "Max_Z_Effective_Date"],
    ascending=[True, False, False]
)
records = raw_sorted.drop_duplicates(subset=["open862_key"], keep="first").copy()
print(f"True unique backorder records: {len(records):,}  (down from {len(raw):,} exploded rows)")


True unique backorder records: 5,261  (down from 89,705 exploded rows)


In [3]:
core_cols = ["open862_key","RELID","ITMID","ITMDESC","VNDID","VNDNAM","H_PLT","PRPCDE","PLNRCD",
             "FSTQTY","Delta Open Qty","OpenQty","SHIP QTY","Status Column","Shipment Status",
             "Aged Category","Required Ship Date","Transit Date","z_loaded_datetime"]
records = records[core_cols].rename(columns={
    "H_PLT": "Plant", "VNDID": "VendorID", "VNDNAM": "VendorName", "ITMID": "ItemID",
    "ITMDESC": "ItemDesc", "RELID": "ReleaseID", "PLNRCD": "PlannerCode", "FSTQTY": "RequiredQty",
    "Delta Open Qty": "OpenQtyDelta", "OpenQty": "OpenQtyRaw", "SHIP QTY": "ShipQty",
    "Status Column": "Status", "Shipment Status": "ShipmentStatus", "Aged Category": "AgedCategory",
    "Required Ship Date": "RequiredShipDate", "Transit Date": "TransitDate",
})

# Join real supplier master (name, owner) from the ASN Dashboard export
suppliers = pd.read_excel(ASN_PATH, sheet_name="Suppliers")
suppliers["Supplier ID"] = suppliers["Supplier ID"].astype(str).str.strip()
records["VendorID"] = records["VendorID"].astype(str).str.strip()
merged = records.merge(
    suppliers[["Supplier ID", "Supplier Name", "SCA", "SCA_Name", "Supplier Location"]],
    left_on="VendorID", right_on="Supplier ID", how="left"
)
print(f"Supplier match rate: {merged['Supplier Name'].notna().mean():.0%}  <- real data gap to flag")

as_of = merged["z_loaded_datetime"].max()
merged["DelayDays"] = (as_of - merged["RequiredShipDate"]).dt.days
merged.loc[merged["Status"] != "Past Due", "DelayDays"] = 0
merged = merged[merged["Status"] == "Past Due"].copy()
print(f"Genuinely past-due records: {len(merged):,}")
merged.head(3)


Supplier match rate: 67%  <- real data gap to flag
Genuinely past-due records: 4,977


,open862_key,ReleaseID,ItemID,ItemDesc,VendorID,VendorName,Plant,PRPCDE,PlannerCode,RequiredQty,...,AgedCategory,RequiredShipDate,TransitDate,z_loaded_datetime,Supplier ID,Supplier Name,SCA,SCA_Name,Supplier Location,DelayDays
0,1.0,50926093.0,12600167,"HDW,MISC,BALL,.708DIA",250784,SKF AUTOMOTIVE USA INC,PR,0.0,2.0,600.0,...,AGED,2025-09-30,2025-09-26,2026-04-22 02:20:17,250784,SKF,Rashad,Rashad,NaN,222
1,2.0,50926093.0,8873,"HDW,MISC,X",250784,SKF AUTOMOTIVE USA INC,PR,0.0,2.0,3840.0,...,AGED,2025-09-30,2025-09-26,2026-04-22 02:20:17,250784,SKF,Rashad,Rashad,NaN,222
2,3.0,51203026.0,34975-02C,"INT SHFTR MECH CMPNT,PIN RATCHET ARM",243070,IM GEARS PVT LTD,PR,0.0,2.0,1183.0,...,AGED,2025-12-05,2025-12-03,2026-04-22 02:20:17,243070,IM GEARS,Rashad,Rashad,India,156


## Step 2 — Generate Mock SAP-Style Fields

Cost, revenue impact, and MRP review status don't exist in the Power BI export — they live in SAP.
Every mock value is generated **deterministically from the real record's own identifiers** (via hashing),
so re-running this notebook always produces the same values for the same record — not random noise.

In [4]:
import hashlib

def stable_hash(s):
    return int(hashlib.md5(str(s).encode()).hexdigest(), 16)

def base_cost(desc):
    """Unit cost tier by item-description keyword — fasteners cheap, subassemblies expensive."""
    d = str(desc).upper()
    if any(k in d for k in ["SCREW","NUT","WASHER","RING","KEY","CLIP","PIN","BADGE","MEDALLION","HDW"]):
        lo, hi = 0.15, 4.0
    elif any(k in d for k in ["SEAL","GUIDE","COLLAR","SEAT","VALVE","BRACKET","MOUNT","SENSOR"]):
        lo, hi = 3.0, 45.0
    elif any(k in d for k in ["GEAR","SHAFT","PUMP","MODULE","ECU","HARNESS","ASSY","ASSEMBLY"]):
        lo, hi = 40.0, 350.0
    elif any(k in d for k in ["FRAME","ENGINE","TRANSMISSION","WHEEL"]):
        lo, hi = 200.0, 1800.0
    else:
        lo, hi = 2.0, 60.0
    h = stable_hash(desc) % 10000 / 10000
    return round(lo + h * (hi - lo), 2)

df = merged.copy()
rng = np.random.default_rng(42)
df["UnitCost_STPRS"] = df["ItemDesc"].apply(base_cost)
df["SalesPricePerUnit_Mock"] = (df["UnitCost_STPRS"] * rng.uniform(1.8, 3.2, size=len(df))).round(2)
df["OpenQtyDelta"] = df["OpenQtyDelta"].fillna(0)
df["MaterialValueAtRisk"] = (df["OpenQtyDelta"] * df["UnitCost_STPRS"]).round(2)
df["EstRevenueImpact"] = (df["OpenQtyDelta"] * df["SalesPricePerUnit_Mock"]).round(2)

# MRP review status (mock DISPO workflow)
mrp_controllers = ["J. Alvarez","M. Chen","R. Patel","K. Novak","S. Odom","T. Brooks"]
df["MRPController_DISPO"] = [mrp_controllers[stable_hash(str(r)) % len(mrp_controllers)] for r in df["PlannerCode"]]
df["MRPReviewed_Flag"] = np.where(rng.random(len(df)) < 0.55, "Y", "N")
review_days_ago = rng.integers(0, 21, size=len(df))
today = pd.Timestamp("2026-08-17")
df["MRPReviewDate"] = np.where(df["MRPReviewed_Flag"] == "Y",
                                (today - pd.to_timedelta(review_days_ago, unit="D")).astype(str), "")
action_options = ["Expedite requested","Split shipment negotiated","Escalated to supplier QBR",
                  "Awaiting supplier confirmation","Alternate source being evaluated","No action yet"]
df["LastActionTaken"] = [action_options[stable_hash(str(r) + "a") % len(action_options)] if f == "Y" else "No action yet"
                          for r, f in zip(df["open862_key"], df["MRPReviewed_Flag"])]

reason_codes = {"RM01":"Raw material shortage at supplier","CAP02":"Supplier capacity constraint",
                "QA03":"Quality hold / rework at supplier","LOG04":"Logistics/carrier delay",
                "ENG05":"Engineering change in process","LAB06":"Labor shortage at supplier plant"}
codes = list(reason_codes.keys())
df["DelayReasonCode"] = [codes[stable_hash(str(r) + "r") % len(codes)] for r in df["open862_key"]]
df["DelayReasonDesc"] = df["DelayReasonCode"].map(reason_codes)

df[["ItemDesc","UnitCost_STPRS","EstRevenueImpact","MRPReviewed_Flag","DelayReasonDesc"]].head(5)


# Alternate supplier suggestion (mock) - placeholder for the future Buyer's Console
alt_pool = df[["VendorID", "VendorName"]].drop_duplicates()
def pick_alt(row):
    candidates = alt_pool[alt_pool["VendorID"] != row["VendorID"]]
    if len(candidates) == 0:
        return pd.Series([None, None])
    idx = stable_hash(str(row["open862_key"]) + "alt") % len(candidates)
    c = candidates.iloc[idx]
    return pd.Series([c["VendorID"], c["VendorName"]])

df[["AltSupplierID_Mock", "AltSupplierName_Mock"]] = df.apply(pick_alt, axis=1)
df["AltSupplier_LeadTimeDays_Mock"] = rng.integers(5, 45, size=len(df))
df["AltSupplier_OTIF_Mock"] = rng.integers(70, 99, size=len(df))


## Step 3 — Prioritization Scoring Model

Revenue impact, delay days, and material value live on incompatible scales (dollars in the single digits
to millions; delay in single days to hundreds). Each is converted to a **percentile rank** before being
combined, so no single raw scale dominates the score.

In [5]:
def pct_rank(s):
    return s.rank(pct=True, method="average") * 100

df["_rev_pct"] = pct_rank(df["EstRevenueImpact"])
df["_delay_pct"] = pct_rank(df["DelayDays"])
df["_matval_pct"] = pct_rank(df["MaterialValueAtRisk"])
df["_unreviewed_bonus"] = np.where(df["MRPReviewed_Flag"] == "N", 100, 0)

WEIGHTS = {"_rev_pct": 0.40, "_delay_pct": 0.25, "_matval_pct": 0.20, "_unreviewed_bonus": 0.15}
df["PriorityScore"] = sum(df[c] * w for c, w in WEIGHTS.items()).round(1)

def tier(score):
    if score >= 75: return "Critical"
    if score >= 55: return "High"
    if score >= 35: return "Medium"
    return "Low"

df["PriorityTier"] = df["PriorityScore"].apply(tier)
print(df["PriorityTier"].value_counts())

top5 = df.sort_values("PriorityScore", ascending=False).head(5)
top5[["ItemDesc","Plant","VendorName","DelayDays","EstRevenueImpact","PriorityScore","PriorityTier"]]


PriorityTier
Medium      1660
High        1449
Low         1303
Critical     565
Name: count, dtype: int64


,ItemDesc,Plant,VendorName,DelayDays,EstRevenueImpact,PriorityScore,PriorityTier
617,"STATOR ASSY,STACK,1.2 INCH",PR,THE INTEC GROUP INC,237,849530.88,97.6,Critical
619,"STATOR ASSY,STACK,1.2 INCH",PR,THE INTEC GROUP INC,237,805593.60,97.6,Critical
803,"STATOR ASSY,STACK,1.2 INCH",PR,THE INTEC GROUP INC,195,1630444.80,97.5,Critical
751,"STATOR ASSY,STACK,1.2 INCH",PR,THE INTEC GROUP INC,216,1116057.60,97.4,Critical
748,"STATOR ASSY,STACK,1.2 INCH",PR,THE INTEC GROUP INC,216,1075472.64,97.1,Critical


## Step 4 — AI Drill-Down Logic

Selecting a record produces a written summary and answers free-form questions. This is **template-based**,
not a live language-model call — every value substituted below is real or mock *data*, not generated text.
This design keeps the demo 100% reliable with no external API dependency; the same record structure is
exactly what a live LLM prompt would need, making it a direct swap-in later.

In [6]:
import assistant as ai

df.to_csv("_scored_backorders.csv", index=False)
scored = ai.load_data("_scored_backorders.csv")
top_record = ai.get_record(scored, scored.sort_values("PriorityScore", ascending=False).iloc[0]["open862_key"])

print(ai.summarize(top_record))


**STATOR ASSY,STACK,1.2 INCH** (Item 29900056A) from **THE INTEC GROUP INC**, Plant PR, is currently **237 days past due** (Critical priority, score 98/100).

Open quantity: 1,152 units short of the 1,920 required. Required ship date was 2025-09-15.

Likely delay reason: **Supplier capacity constraint** (code CAP02).

MRP review status: this backorder has NOT been reviewed by the assigned controller (S. Odom). No action has been logged yet.

Estimated business impact if not resolved: **$849,531** in potential revenue exposure (material value at risk: $371,773).

Recommended next action: **Escalate immediately** to the MRP controller for review — this is high-impact and has not been looked at yet.

Alternate supplier option: **THYSSENKRUPP PRESTA DANVILLE LLC** (mock lead time: 12 days, mock OTIF score: 78%).


In [7]:
# Free-form Q&A — matched by question intent, not exact phrasing.
# Stress-tested against 19 naturally-phrased variants before the demo.
for q in [
    "What was the reason for the material delay?",
    "why is this late",                          # rephrased, not the exact example
    "Has this backorder been reviewed by the MRP Controller?",
    "who is the current vendor",                 # rephrased
    "What is the expected sales/revenue impact if this order is not delivered on time?",
]:
    print(f"Q: {q}\nA: {ai.answer_question(top_record, q)}\n")


Q: What was the reason for the material delay?
A: The likely reason for the delay is: Supplier capacity constraint (code CAP02).

Q: why is this late
A: The likely reason for the delay is: Supplier capacity constraint (code CAP02).

Q: Has this backorder been reviewed by the MRP Controller?
A: No — this backorder has not yet been reviewed. The assigned MRP controller is S. Odom.

Q: who is the current vendor
A: The current supplier for this backorder is THE INTEC GROUP INC (Plant PR).

Q: What is the expected sales/revenue impact if this order is not delivered on time?
A: Estimated revenue impact if not delivered on time: $849,531. Material value at risk: $371,773.



## Summary

| Step | Technique |
|---|---|
| 1. Clean | True-grain recovery from an exploded/obfuscated row structure |
| 2. Enrich | Deterministic mock-data generation, keyed to real record identifiers |
| 3. Score | Feature engineering + percentile normalization + weighted multi-factor scoring |
| 4. Explain | Template-based natural-language generation + intent-keyword Q&A matching |

This is the exact logic running behind the interactive HTML demo — the demo simply renders this same
pipeline's output in a clickable UI instead of a notebook cell.